# Final study, notebook B: M2AC and SAC

**Kaggle account: silvergjeka01.** This notebook trains **M2AC for seeds 0-3 at the same time, then SAC for seeds 0-3 (8 trainings, 4 at a time)**. Notebook A (account nouradon) trains MACURA and MBPO on the same seeds and settings.
The results of both are merged on the computer by `notebooks/final_merge.ipynb` (tables, figures, tests, videos).

**Settings (the same in A and B, decided before the runs):**
- Task **race3** (race2 with a deadlier chute: up to 50% lift lost instead of 35%), **50,000 real steps**
  (5,000 random warm-up + 45,000 learning), **seeds 0, 1, 2, 3**, the race2 paper protocol.
- **Equal training updates**: MBPO and M2AC do **12 SAC updates per real step** = MACURA's average in the race2
  pilot (11.7); MACURA keeps its adaptive updates (up to 16); SAC does 1 (model-free).
- Exploration as in the MACURA paper (disclosed): MACURA pink noise, MBPO and M2AC without noise, SAC samples its policy.
- If the race3 test's decision rule chooses race2 instead, set `TASK = "race2"` in section 1 (everything else stays).

Time: about 3-4 h (estimate).

## 1. Settings

In [ ]:
import os, sys, time, subprocess

NOTEBOOK = "B"                    # A = MACURA + MBPO (nouradon) | B = M2AC + SAC (silvergjeka01)
SEEDS    = [0, 1, 2, 3]
TASK     = "race3"
STEPS    = 50000                  # per training: 5,000 warm-up + 45,000 learning
TOGETHER = ["m2ac"]   # trained at the same time, for every seed
AFTER    = ["sac"]              # trained when a slot is free
UPDATES  = 12                     # SAC updates per real step for MBPO and M2AC (MACURA: adaptive, up to 16)
MAX_JOBS = None                   # trainings at the same time; None = min(4, CPU cores)
REPO, BRANCH = "silvergjeka22/macura-drone", "race3"
ROOT = "/kaggle/working/macura-drone"
OUT  = "/kaggle/working/runs"

T0 = time.time()
DEADLINE = T0 + 11.0 * 3600       # Kaggle stops a session at 12 h and then saves nothing: training stops at 11 h
print("=" * 78)
print(f"  NOTEBOOK {NOTEBOOK}   |   SEEDS {SEEDS}   |   task {TASK}   |   {STEPS:,} steps")
print(f"  algorithms: {', '.join(TOGETHER + AFTER)}   |   MBPO / M2AC {UPDATES} updates per step")
print("=" * 78)
os.environ.update({"MACURA_TASK": TASK, "MACURA_STEPS": str(STEPS), "MACURA_SEEDS": " ".join(map(str, SEEDS)),
                   "MACURA_ALGOS": " ".join(TOGETHER + AFTER), "MBPO_UTD": str(UPDATES), "M2AC_UTD": str(UPDATES),
                   "MACURA_OUTPUT_ROOT": OUT, "MACURA_DEADLINE": str(DEADLINE)})

token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)
print("code:", subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
import src.config.config as cfg
from src.analysis import results as R
from src.viz import figures as F

N_JOBS = MAX_JOBS or min(4, os.cpu_count() or 1)
N_GPU = torch.cuda.device_count()
print(f"config: task {cfg.TASK} | seeds {cfg.SEEDS} | max lift loss {cfg.ENV['vrs_loss']:.0%} | "
      f"updates per step: MBPO {cfg.ROLLOUT['mbpo']['fixed_gradient_steps']}, M2AC {cfg.ROLLOUT['m2ac']['fixed_gradient_steps']}, "
      f"MACURA up to {cfg.ROLLOUT['macura']['gradient_steps_max']} (xi {cfg.ROLLOUT['macura']['xi']}) | "
      f"warm-up {cfg.WARMUP_RANDOM_STEPS}")
print(f"machine: {os.cpu_count()} CPU cores, {N_GPU} GPU(s) -> {N_JOBS} trainings at the same time")
os.makedirs(OUT, exist_ok=True)
json.dump({"notebook": NOTEBOOK, "seeds": SEEDS, "task": TASK, "steps": STEPS, "algorithms": TOGETHER + AFTER,
           "updates": UPDATES, "started": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime(T0))},
          open(f"{OUT}/RUN_INFO.json", "w"), indent=2)

CHILD = '''
import sys
sys.path.insert(0, {root!r})
from src.config import config as cfg
from src.training.train import train_algo
train_algo({algo!r}, cfg.CFG, cfg.OUTPUT_ROOT, seeds=[{seed}])
'''


def _clock():
    return f"{(time.time() - T0) / 3600:4.1f} h"


def _last_eval(log):
    try:
        lines = [l for l in open(log).read().splitlines() if l.startswith("[") and " step " in l]
        return lines[-1].strip() if lines else "(warm-up)"
    except OSError:
        return "(no log yet)"


def run_jobs(jobs, n_jobs, report_every=900):
    # each training is its own process (one CPU thread each); a new one starts when a slot is free
    os.makedirs(f"{OUT}/joblogs", exist_ok=True)
    pending, running, last = list(jobs), {}, 0.0
    while pending or running:
        while pending and len(running) < n_jobs:
            algo, seed = pending.pop(0)
            if os.path.exists(f"{OUT}/logs/{algo}_seed{seed}.json"):
                print(f"[{_clock()}] {algo} seed {seed}: already finished, reloaded"); continue
            if time.time() > DEADLINE:
                print(f"[{_clock()}] {algo} seed {seed}: no session time left, skipped"); continue
            env = dict(os.environ, MACURA_SEEDS=str(seed), MACURA_ALGOS=algo, PYTHONUNBUFFERED="1",
                       OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
            if N_GPU > 1:
                env["CUDA_VISIBLE_DEVICES"] = str(len(running) % N_GPU)
            log = f"{OUT}/joblogs/{algo}_seed{seed}.txt"
            p = subprocess.Popen([sys.executable, "-u", "-c", CHILD.format(root=ROOT, algo=algo, seed=seed)],
                                 env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[(algo, seed)] = (p, log, time.time())
            print(f"[{_clock()}] started {algo} seed {seed}  (log: joblogs/{algo}_seed{seed}.txt)", flush=True)
        for (algo, seed), (p, log, t0) in list(running.items()):
            if p.poll() is None:
                continue
            del running[(algo, seed)]
            mins = (time.time() - t0) / 60
            if p.returncode == 0:
                final = [l for l in open(log).read().splitlines() if "FINAL" in l]
                print(f"[{_clock()}] finished {algo} seed {seed} in {mins:.0f} min: {final[-1].strip() if final else ''}",
                      flush=True)
            else:
                print(f"[{_clock()}] FAILED {algo} seed {seed} (exit {p.returncode}); last lines:\n"
                      + "\n".join(open(log).read().splitlines()[-15:]), flush=True)
        if running and time.time() - last > report_every:
            last = time.time()
            print(f"[{_clock()}] progress:", flush=True)
            for (algo, seed), (p, log, t0) in running.items():
                print(f"    {_last_eval(log)}", flush=True)
        time.sleep(15)
    print(f"[{_clock()}] all trainings done", flush=True)

## 2. Training
The trainings in `TOGETHER` start for the seeds in order (4 at a time); those in `AFTER` start when a slot is free.
A progress line per training appears every 15 minutes (its latest evaluation); each training's full printout is in
`runs/joblogs/`. Running this cell again reloads finished trainings instead of repeating them.

In [ ]:
JOBS = [(a, s) for s in SEEDS for a in TOGETHER] + [(a, s) for s in SEEDS for a in AFTER]
print("NOTEBOOK", NOTEBOOK, "| SEEDS", SEEDS, "->", len(JOBS), "trainings:", ", ".join(f"{a} seed {s}" for a, s in JOBS))
run_jobs(JOBS, N_JOBS)

## 3. Results of this notebook
A first look only; the full comparison of all four algorithms is made by `notebooks/final_merge.ipynb`.
- **Per seed**: average return while learning and drones broken after the warm-up (the main scores), then the best
  checkpoint's test on 30 fresh scenarios.
- **Updates per real step**, and **the chute**: did a *trained* policy lose lift (second half of training and the
  final test only)?

In [ ]:
RUNS = R.load_runs(OUT)
print("NOTEBOOK", NOTEBOOK, "| results found for SEEDS", sorted({r["seed"] for r in RUNS}),
      "| algorithms", sorted({r["algo"] for r in RUNS}))
missing = [(a, s) for a, s in JOBS if not any(r["algo"] == a and r["seed"] == s for r in RUNS)]
print("MISSING:", missing if missing else "none")
stopped = [f"{r['algo']} seed {r['seed']} at step {r['setup']['stopped_early_at']}" for r in RUNS
           if r.get("setup", {}).get("stopped_early_at")]
print("stopped by the time limit:", stopped if stopped else "none")
display(Markdown("### Per seed\n\n" + R.per_seed_markdown(RUNS)))
display(Markdown(R.summary_markdown(RUNS)))

utd = ["| algorithm | seed | updates per real step (mean) |", "|---|---|---|"]
for r in RUNS:
    utd.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | "
               + (f"{np.mean(r['utd']):.1f}" if r.get("utd") else "1 (model-free)") + " |")
display(Markdown("### Updates per real step\n\n" + "\n".join(utd)))

chute = ["| algorithm | seed | fastest descent, 2nd half (m/s) | time losing lift, 2nd half | test: fastest descent | "
         "test: time losing lift |", "|---|---|---|---|---|---|"]
for r in RUNS:
    late = np.asarray(r["steps"]) >= STEPS / 2
    sink, lift, fe = np.asarray(r["eval_max_sink"])[late], np.asarray(r["eval_liftloss"])[late], r["final_eval"]
    chute.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | {sink.max() if len(sink) else float('nan'):.2f} | "
                 f"{100 * lift.mean() if len(lift) else float('nan'):.1f}% | {fe.get('eval_max_sink', float('nan')):.2f} | "
                 f"{100 * fe.get('eval_liftloss', 0):.1f}% |")
display(Markdown("### The chute (lift loss starts at 1.2 m/s)\n\n" + "\n".join(chute)))

F.plot_learning_curves(RUNS, None, f"{OUT}/plots/learning_curves_{NOTEBOOK}.png"); plt.show()
open(f"{OUT}/summary_{NOTEBOOK}.md", "w").write("\n\n".join(
    [f"# Final study, notebook {NOTEBOOK}, seeds {SEEDS}", R.per_seed_markdown(RUNS), "\n".join(utd), "\n".join(chute)]))
print("saved:", f"{OUT}/summary_{NOTEBOOK}.md")

## 4. Outputs
`/kaggle/working/runs`: `RUN_INFO.json` (which notebook, seeds, settings), `logs/` (one JSON per algorithm and seed),
`checkpoints/` (best policy + world model per run), `joblogs/`, `plots/`, `summary_B.md`.
Download both notebooks with `./finals.sh get` (each into its own new folder, nothing is overwritten), then open
`notebooks/final_merge.ipynb`.